# Neural Network Model Training

Train, evaluate, and save an `MLPRegressor` for concrete compressive-strength prediction.

<h3><b>1. Import libraries and configure project paths <b><h3>

In [8]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'data' / 'processed' / 'concrete_data_cleaned.csv').exists()
)

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'concrete_data_cleaned.csv'
MODEL_DIR = PROJECT_ROOT / 'models' / 'trained'
PREPROCESS_DIR = PROJECT_ROOT / 'models' / 'preprocessing'
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'evaluation'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)

Project root: d:\My Projects\Concrete-Compressive-Strength-Predictor


<h3><b>2. Load the cleaned dataset <b><h3>

In [9]:
df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
display(df.head())

Dataset shape: (911, 9)


,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
1,266.0,114.0,0.0,228.0,0.0,932.0,670.0,90,47.03
2,380.0,95.0,0.0,228.0,0.0,932.0,594.0,28,36.45
3,266.0,114.0,0.0,228.0,0.0,932.0,670.0,28,45.85
4,475.0,0.0,0.0,228.0,0.0,932.0,594.0,28,39.29


<h3><b>3. Separate features and target <b><h3>

In [10]:
target_column = 'compressive_strength'
X = df.drop(columns=[target_column])
y = df[target_column]

print('Feature shape:', X.shape)
print('Target shape:', y.shape)

Feature shape: (911, 8)
Target shape: (911,)


<h3><b>4. Split the data and scale features <b><h3>

<h6>Scaling is important for neural networks because it places all input features on a comparable range.<h6>

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Training samples:', X_train.shape[0])
print('Test samples:', X_test.shape[0])

Training samples: 728
Test samples: 183


<h3><b>7. Train the neural network <b><h3>

In [12]:
neural_network = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    random_state=42,
    max_iter=3000,
    early_stopping=True,
)

neural_network.fit(X_train_scaled, y_train)
print('Neural network training completed.')

Neural network training completed.


<h3><b>6. Evaluate the trained model <b><h3>

In [13]:
y_pred = neural_network.predict(X_test_scaled)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f'R²: {r2:.4f}')
print(f'MAE: {mae:.4f}')
print(f'RMSE: {rmse:.4f}')

R²: 0.6144
MAE: 8.0278
RMSE: 10.0298


<h3><b>7. Save the trained model, scaler, and metrics <b><h3>

In [14]:
model_file = MODEL_DIR / 'neural_network.joblib'
scaler_file = PREPROCESS_DIR / 'neural_network_scaler.joblib'
metrics_file = REPORTS_DIR / 'neural_network_metrics.json'

joblib.dump(neural_network, model_file)
joblib.dump(scaler, scaler_file)

metrics = {
    'model': 'MLPRegressor',
    'r2': float(r2),
    'mae': float(mae),
    'rmse': float(rmse),
    'model_file': str(model_file),
    'scaler_file': str(scaler_file),
}
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Saved model: {model_file}')
print(f'Saved scaler: {scaler_file}')
print(f'Saved metrics: {metrics_file}')

Saved model: d:\My Projects\Concrete-Compressive-Strength-Predictor\models\trained\neural_network.joblib
Saved scaler: d:\My Projects\Concrete-Compressive-Strength-Predictor\models\preprocessing\neural_network_scaler.joblib
Saved metrics: d:\My Projects\Concrete-Compressive-Strength-Predictor\reports\evaluation\neural_network_metrics.json
